In [ ]:
import logging
import os
import re
from pathlib import Path
from typing import List, Tuple

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from napistu.utils import load_pickle, save_pickle
from napistu.constants import SBML_DFS
from napistu.ontologies.constants import ONTOLOGIES
from napistu.network.constants import NAPISTU_GRAPH_VERTICES

from napistu_torch.load.constants import DEFAULT_ARTIFACTS_NAMES
from napistu_torch.load.foundation_models import (
    AttentionPatternsInputs,
    FoundationModels,
    aggregate_embedding_comparisons_over_categories,
    validate_embedding_comparisons_settings,
    _get_disk_name,
)
from napistu_torch.load.constants import FM_EDGELIST, MODEL_NICE_NAMES
from napistu_torch.napistu_data_store import NapistuDataStore
from napistu_torch.utils.napistu_utils import map_identifiers_to_vertex_names
from napistu_torch.utils.pd_utils import reorder_multindex_by_categorical_and_numeric
from napistu_torch.utils.tensor_utils import compute_correlation_matrix
from napistu_torch.visualization.heatmaps import plot_heatmap

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s')
logger = logging.getLogger(__name__)

CONSENSUS_METHOD = "sum"
BY_ABSOLUTE_VALUE = False
VERBOSE = False
TOP_K = 10000
IGNORE_SELF_ATTENTION = True
OVERWRITE = False

PROJECT_DIR = Path("~/Desktop/DATA/sc_foundation_models").expanduser()
CACHE_DIR = PROJECT_DIR / "cache"
EMBEDDING_DATASET = "efthymiou2025"
MODEL_OUTPUTS_DIR = PROJECT_DIR / "model_outputs" # where extracted foundation model weights 

INCLUDE_SCGPT = True
SCGPT_MODEL_NAME = "scGPT" # process with and without scGPT since scGPT only uses the top 1200 highly variable genes
IGNORE_CATEGORIES_WITH = ["unknown"]

def to_filename(s: str) -> str:
    s = re.sub(r'[^\w\s-]', '', s)
    s = re.sub(r'\s+', '_', s)
    return s.strip('-_')


def get_cache_path(dataset: str, category: str, include_scgpt: bool) -> Path:
    scgpt_flag = "with_scgpt" if include_scgpt else "without_scgpt"
    return CACHE_DIR / f"comparisons_{dataset}_{to_filename(category)}_{scgpt_flag}.pkl"


def get_model_prefixes(include_scgpt: bool = True) -> List[str]:
    return [
        _get_disk_name(model_name, model_variant)
        for model_name, model_variant in MODEL_NICE_NAMES
        if include_scgpt or model_name != SCGPT_MODEL_NAME
    ]

def get_all_categories(
    output_dir: str = MODEL_OUTPUTS_DIR,
    embedding_dataset: str = EMBEDDING_DATASET,
    ignore_categories_with: List[str] = IGNORE_CATEGORIES_WITH,
) -> List[str]:
    fms = FoundationModels.load_multiple(output_dir, get_model_prefixes(), verbose = False)

    first_model_keys = fms.models[0].dataset_gene_embeddings.get(embedding_dataset).keys()
    other_model_keys = [set(x.dataset_gene_embeddings.get(embedding_dataset).keys()) for x in fms.models[1:]]

    all_categories = [k for k in first_model_keys if all(k in s for s in other_model_keys)]

    # filter categories to exclude categories containing substrings from IGNORE_CATEGORIES_WITH
    valid_categories = [cat for cat in all_categories if not any(substring in cat for substring in ignore_categories_with)]

    return valid_categories

def get_model_comparison_metadata(include_scgpt):

    model_comparison_metadata = dict()
    
    models = FoundationModels.load_multiple(MODEL_OUTPUTS_DIR, get_model_prefixes(include_scgpt), verbose=False)
    model_comparison_metadata["model_order"] = models.model_names

    model_metadata_summary = models.get_summary().rename(
        columns = {
            "full_name": "model",
            "model": "model type",
            "variant": "model variant",
            "embed_dim": "# embedding dim",
            "n_layers": "# layers",
            "n_heads": "# heads",
            "parameter_count": "# parameters",
        }
    )
    # genes in the dataset embedding
    model_metadata_summary["# genes in dataset embedding"] = [x.dataset_gene_embeddings[EMBEDDING_DATASET].n_common_genes for x in models.models]
    model_comparison_metadata["model_metadata_summary"] = model_metadata_summary

    a_category = list(models.models[0].dataset_gene_embeddings[EMBEDDING_DATASET].data.keys())[0]
    attended_embeddings = AttentionPatternsInputs.from_expression(models, EMBEDDING_DATASET, a_category, verbose=False)
    model_comparison_metadata["gene_ids"] = attended_embeddings.common_gene_ids

    return model_comparison_metadata

def add_vertex_names_to_edgelist(
    edgelist: pd.DataFrame,
    gene_to_vertex_map: pd.DataFrame
) -> pd.DataFrame:

    """
    Add from_vertex and to_vertex attributes by mapping an edgelist with ensembl gene include_scgpt

    Parameters
    ----------
    edgelist : pd.DataFrame
        An edgelist with from_gene and to_gene columns
    gene_to_vertex_map : pd.DataFrame
        A dataframe with ensembl_gene and name columns

    Returns
    -------
    pd.DataFrame
        An edgelist with from_vertex and to_vertex columns
    """

    top_k_with_ids = (
        edgelist
        .merge(
            (
                gene_to_vertex_map
                .rename(columns = {ONTOLOGIES.ENSEMBL_GENE : FM_EDGELIST.FROM_GENE, NAPISTU_GRAPH_VERTICES.NAME : "from_vertex"})
                .drop(columns = [SBML_DFS.S_ID])
            ),
            on = FM_EDGELIST.FROM_GENE,
            how = "left"
        )
        .merge(
            gene_to_vertex_map
            .rename(columns = {ONTOLOGIES.ENSEMBL_GENE : FM_EDGELIST.TO_GENE, NAPISTU_GRAPH_VERTICES.NAME : "to_vertex"})
            .drop(columns = [SBML_DFS.S_ID]),
            on = FM_EDGELIST.TO_GENE,
            how = "left"
        )
    )

    # drop NAs and log
    invalid_edges = top_k_with_ids.isna().any(axis = 1)
    if invalid_edges.sum() > 0:
        percent_invalid = (invalid_edges.sum() / len(top_k_with_ids)) * 100
        logger.warning(f"Dropping {invalid_edges.sum()} edges ({percent_invalid:.2f}%) which could not be mapped to vertices")

    return top_k_with_ids.loc[not invalid_edges]


bwy = LinearSegmentedColormap.from_list(
    "blue_white_yellow",
    ["#2166AC", "#FFFFFF", "#F5C800"]  # strong blue → white → bright yellow
)


In [ ]:
STORE_DIR = os.path.expanduser("~/Desktop/EXPERIMENTS/.store")
napistu_data_store = NapistuDataStore(STORE_DIR)
species_identifiers = (
    napistu_data_store.load_pandas_df(DEFAULT_ARTIFACTS_NAMES.SPECIES_IDENTIFIERS)
    .query("bqb in @BQB_DEFINING_ATTRS_LOOSE") # filter complexes (which would could contain a protein with a BQB_HAS_PART qualifier)
)
# map from vertex names (compartmentalized species IDs) to species ids (sids)
name_to_sid_map = napistu_data_store.load_pandas_df(DEFAULT_ARTIFACTS_NAMES.NAME_TO_SID_MAP).reset_index()


EXPECTED_NAME_TO_SID_MAP_COLUMNS = {SBML_DFS.S_ID, NAPISTU_GRAPH_VERTICES.NAME}


In [ ]:
all_categories = get_all_categories()

runs: List[Tuple[str, bool]] = [
    (category, include_scgpt)
    for category in all_categories
    for include_scgpt in (True, False)
]

for category, include_scgpt in runs:
    cache_path = get_cache_path(EMBEDDING_DATASET, category, include_scgpt)

    if cache_path.is_file() and not OVERWRITE:
        if VERBOSE:
            logger.info(f"Skipping {category} (include_scgpt={include_scgpt}), cache exists at {cache_path}")
        continue

    logger.info(f"Running {category} (include_scgpt={include_scgpt})")

    model_prefixes = get_model_prefixes(include_scgpt)
    models_subset = FoundationModels.load_multiple(MODEL_OUTPUTS_DIR, model_prefixes, verbose=False)
    
    attended_embeddings = AttentionPatternsInputs.from_expression(models_subset, EMBEDDING_DATASET, category)

    comparisons = attended_embeddings.compare(
        top_k = TOP_K,
        consensus_method = CONSENSUS_METHOD,
        by_absolute_value = BY_ABSOLUTE_VALUE,
        ignore_self_attention = IGNORE_SELF_ATTENTION,
        verbose = VERBOSE
    )

    save_pickle(cache_path, comparisons)

    del attended_embeddings, comparisons, models_subset

In [ ]:
model_summaries = dict()
for include_scgpt in (True, False):
    # summaries of the models underconsideration - model size, vocabulary, etc.
    metadata = get_model_comparison_metadata(include_scgpt)
    model_order = metadata["model_order"]

    # load comparisons for each category
    category_summaries = dict()
    for category in all_categories:

        cache_path = get_cache_path(EMBEDDING_DATASET, category, include_scgpt)
        category_comparisons = load_pickle(cache_path)
        
        # validate that the comparisons were generated using the expected settings
        validate_embedding_comparisons_settings(category_comparisons, TOP_K, CONSENSUS_METHOD, BY_ABSOLUTE_VALUE, IGNORE_SELF_ATTENTION, model_order)
        category_summaries[category] = load_pickle(cache_path)

    # aggregate comparisons over categories
    comparisons = aggregate_embedding_comparisons_over_categories(category_summaries)
    
    summaries = {
        "model_comparison_metadata": metadata,
        "comparisons": comparisons
    }
    
    model_summaries[include_scgpt] = summaries

In [ ]:
# select summaries that will be used for the majority of the plots - some will compare results across model sets with and without scGPT
comparisons = model_summaries[INCLUDE_SCGPT]["comparisons"]
model_order = model_summaries[INCLUDE_SCGPT]["model_comparison_metadata"]["model_order"]

# remove some unused information from the summary version which isn't referenced by INCLUDE_SCGPT
UNNEEDED_KEYS = ["gene_embedding_correlations", "model_layer_correlations", "cross_model_consensus_top_attentions", "cross_model_consensus_top_attentions_rank_agreement"]
model_summaries[not INCLUDE_SCGPT]["comparisons"] = {k: v for k, v in model_summaries[not INCLUDE_SCGPT]["comparisons"].items() if k not in UNNEEDED_KEYS}

## Compare embeddings

To assess whether the embeddings are organizing genes in a similar way, I directly compared them. Since their embedding dimensions differ I decided to generate a gene-gene distance matrix (using cosine distance) for each embedding and then I compared all pairs of distance matrices using Spearman correlation.

The highest concordance is between scGPT and scPRINT (rho ~ 0.28) while AIDO.Cell's gene embeddings show low concordance with the other models, and between the different AIDO.Cell models (i.e., the 3M, 10M, and 100M cell models). This may just be a feature of AIDO.Cell's architecture since it uses gene-level positional embeddings which may not actual learn gene-gene similarity.

In [ ]:
embeddings_comparisons = comparisons["gene_embedding_correlations"]

wide_embeddings_comparisons = (
    pd.DataFrame(list(embeddings_comparisons.items()), columns=['model', 'rho'])
    .assign(
        model1=lambda df: df['model'].str.split("_vs_", expand=True)[0],
        model2=lambda df: df['model'].str.split("_vs_", expand=True)[1]
    )
    .pivot(index='model2', columns='model1', values='rho')
    .reindex(index=model_order, columns=model_order)
    .dropna(axis=0, how = "all")
    .dropna(axis=1, how = "all")
)

In [ ]:
plot_heatmap(
    wide_embeddings_comparisons,
    mask_upper_triangle=True,
    vmax=1,
    vmin=0,
    cbar_label='Spearman ρ',
    suptitle="Embedding similarity",
    title="Based on the cross-model correlation of within model\ngene-gene cosine similarity",
    title_size=12,
    square=True,
    title_fontstyle="italic",
    title_fontweight="normal",
)

plt.show()

## Compare attention

Next, I want to compare the gene-gene attention probabilities (the softmax, makes these a probability distribution for each column) to see how pairs of genes are attending to each other.

### Working with attention summaries

Since Foundation models all use Transformer-based architectural with multiple multi-headed attention layers we can summarize the attention mechanisms either layer-wise or as an aggregate over all layers.

#### From a single layers

At the level of an individual layer, a couple of useful summaries are:

- the raw attention patterns: (gene, gene)
- top-k attention pairs

```python
A_MODEL = "scPRINT (small)"
A_LAYER = 3
ae = attended_embeddings[A_MODEL]

# calculate one layer's attention pattern
attention = ae.compute_attention(layer_idx = A_LAYER, apply_softmax = False)
print(attention.shape)

# pulling out its top-k attention pairs
top_k_attention_edges = ae.get_top_attentions(
    k = TOP_K,
    layer_indices = A_LAYER,
    apply_softmax = False,
    by_absolute_value = BY_ABSOLUTE_VALUE,
)

display(top_k_attention_edges.head())
cleanup_tensors(attention, top_k_attention_edges)
```

Either of these summaries can be rolled up across layers but before doing so we can evaluate whether cross-layer summaries are loosly consistent.

To explore this I'll pull out the top-attention pairs for each layer and compare them across the layers within the same model.

For a single model this can be done with:

```python
# get top-k attention edges for each layer
top_k_attention_edges = (
    attended_embeddings[A_MODEL]
    .get_top_attentions(
        k = 10000,
        by_absolute_value = BY_ABSOLUTE_VALUE,
    )
)

# (gene x gene) summary as attention[arg-max(|attention|)]
(
    attended_embeddings[A_MODEL]
    .compute_consensus_attention(
        consensus_method = CONSENSUS_METHOD
    )
)
```

But, the `FoundationModels` class has convenience methods for aggregating summaries from multiple models.

Here, we'll use the `FoundationModels`'s `get_top_attentions()` to pull out each model x layer's top-K attention scores. A similar method, which will be used later, `get_max_attentions` can be used to create a 3D Tensor of each model's (gene x gene) consensus attention scores.


## Within model - across layers

### Looking at top attention pairs from each layer and model

1. define attention probabilities for each layer (I x I). Where is I is the length and ordering of `common_identifiers`
2. select the N greatest attention probabilities from each layer returning the row and column indices and the attention probability
3. aggregate across all layers and models and store results as a pd.DataFrame
4. rename row/column indices as ensembl genes using `common_identifiers`
5. for each model cound distinct row (from) and column (to) pairs
6. pivot so there is 1 row per from-to edge and columns represent counts of each pair in the top attentions of each model




In [ ]:
model_layer_correlations = comparisons["model_layer_correlations"]

fig, axes = plt.subplots(2, 4, figsize=(20, 12), gridspec_kw={"width_ratios": [1, 1, 1, 1.3]})
axes = axes.flatten()

for idx, model_name in enumerate(model_order):

    data = model_layer_correlations[model_name]
    annot_size = 15 - 2*np.sqrt(data.shape[0])
    
    plot_heatmap(
        data,
        row_labels = range(data.shape[0]),
        title=model_name,
        cbar=False,
        fmt='.2f',
        vmax=0.5,
        vmin=-0.5,
        cbar_label='Spearman ρ',
        cmap=bwy,
        mask_upper_triangle=True,
        square=True, 
        title_size=22,
        annot_size=annot_size,
        title_fontstyle="italic",
        ax=axes[idx],
    )

plt.suptitle("Within model layer x layer Spearman correlations using all top attentions", fontsize=24, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


In [ ]:
model_layer_rank_agreement = comparisons["model_layer_rank_agreement"]

N_COLUMNS = 4
fig, axes = plt.subplots(2, N_COLUMNS, figsize=(20, 12))
axes = axes.flatten()

for idx, model_name in enumerate(model_order):

    conditional_quantiles = model_layer_rank_agreement[model_name].pivot_table(index = "query_layer", columns = "eval_layer", values = "median_quantile", aggfunc = "median")

    x_title = "Evaluation layer" if idx >= N_COLUMNS else None
    y_title = f"Top {TOP_K} layer" if idx % N_COLUMNS == 0 else None
    annot_size = 15 - 2*np.sqrt(conditional_quantiles.shape[0])
    
    plot_heatmap(
        conditional_quantiles,
        row_labels = range(conditional_quantiles.shape[0]),
        title=model_name,
        xlabel=x_title,
        ylabel=y_title,
        cbar=False,
        cmap="viridis",
        fmt='.2f',
        vmax=1,
        vmin=0,
        mask_upper_triangle=False,
        square=True,
        title_size=22,
        axis_title_size=18,
        annot_size=annot_size,
        title_fontstyle="italic",
        ax=axes[idx],
    )

plt.suptitle("Median quantiles in an evaluation layer of the topK pairs in a query layer", fontsize=24, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

## Across models (all layers)

In [ ]:
def calculate_cross_model_attention_correlation(cross_model_top_attentions):

    # look at the top attentions for each model/layer across all models and layers
    wide_consistent_attentions = (
        cross_model_top_attentions.pivot_table(
            index = [FM_EDGELIST.FROM_GENE, FM_EDGELIST.TO_GENE],
            columns = [FM_EDGELIST.MODEL, FM_EDGELIST.LAYER],
            values = FM_EDGELIST.ATTENTION
        )
    )

    result = compute_correlation_matrix(wide_consistent_attentions.to_numpy(), verbose=False)
    del wide_consistent_attentions  # explicitly free the ~550K x 70 matrix
    return result

def get_model_layer_labels(cross_model_top_attentions):

    model_layers_df = cross_model_top_attentions[[FM_EDGELIST.MODEL, FM_EDGELIST.LAYER]].drop_duplicates()
    model_layers_df["label"] = model_layers_df.apply(lambda x: f"{x[FM_EDGELIST.MODEL]}-{x[FM_EDGELIST.LAYER]}", axis=1)

    return model_layers_df


cross_model_top_attentions = comparisons["cross_model_x_layer_top_attentions"]
if "category" in cross_model_top_attentions.columns:
    logger.info("Calculating cross-model attention correlation for each category")
    per_category_corr = cross_model_top_attentions.groupby("category").apply(calculate_cross_model_attention_correlation)
    logger.info("Aggregating cross-model attention correlation over categories")
    cross_model_attention_corr = np.median(np.stack(per_category_corr.values), axis=0)
else:
    cross_model_attention_corr = calculate_cross_model_attention_correlation(cross_model_top_attentions)


In [ ]:

model_layers_df = get_model_layer_labels(cross_model_top_attentions)
model_col_values = model_layers_df["model"]
model_layer_labels = model_layers_df["label"].tolist()

# Create mask where models match (True = mask/hide these values)
mask = model_col_values.to_numpy()[:, None] == model_col_values.to_numpy()[None, :]
assert mask.shape == cross_model_attention_corr.shape

### layer x layer attention correlation

In [ ]:
upper_tri_mask = np.triu(np.ones_like(mask, dtype=bool), k=1)
combined_mask = mask | upper_tri_mask

# Find rows and columns that are completely masked (all True)
rows_to_keep = ~combined_mask.all(axis=1)  # Keep rows that have at least one False
cols_to_keep = ~combined_mask.all(axis=0)  # Keep columns that have at least one False

# Filter your data and mask
non_null_cross_model_attention_corr = cross_model_attention_corr[rows_to_keep][:, cols_to_keep]
filtered_mask = combined_mask[rows_to_keep][:, cols_to_keep]

# If you have labels, filter those too
filtered_row_labels = [label for i, label in enumerate(model_layer_labels) if rows_to_keep[i]]
filtered_col_labels = [label for i, label in enumerate(model_layer_labels) if cols_to_keep[i]]

plot_heatmap(
    non_null_cross_model_attention_corr,
    row_labels = filtered_row_labels,
    column_labels = filtered_col_labels,
    suptitle="Cross-model x layer attention consistency",
    title=f"Spearman correlation of attention logits across all\ntop {TOP_K} gene x gene attention pairs.\nWithin-model attention is masked.",
    cmap=bwy,
    cbar=True,
    fmt='.2f',
    vmax=0.25,
    vmin=-0.25,
    cbar_label='Spearman ρ',
    mask = filtered_mask,
    mask_color='lightgray',
    annot=False,
    tick_label_size=5,
    suptitle_size=24,
    title_size=18,
    title_fontstyle="italic",
    title_fontweight="normal",
    figsize=(9, 9)
)

plt.show()

In [ ]:
triplet_df = (
    pd.DataFrame(
        cross_model_attention_corr,
        index = model_layer_labels,
        columns = model_layer_labels
    )
    .stack()
    .reset_index(name='value')
    .rename(columns={'level_0': 'row', 'level_1': 'col'})
    .query('row > col')  # Lower triangle excluding diagonal
    # expand layer metadata
    .merge(model_layers_df.rename(columns = {"model" : "model1", "layer" : "layer1", "label" : "row"}))
    .merge(model_layers_df.rename(columns = {"model" : "model2", "layer" : "layer2", "label" : "col"}))
)

triplet_df.query('model1 != model2').sort_values('value', ascending = True).head(10)

### Layer x layer rank consistency 

In [ ]:
model_x_layer_rank_agreement = comparisons["cross_model_x_layer_rank_agreement"]

# [0,1] low quantiles are more consistent (ranks in attent B | top in A)
conditional_quantiles = (
    model_x_layer_rank_agreement
    .pivot_table(
        index = ["query_model", "query_layer"],
        columns = ["eval_model", "eval_layer"],
        values = "median_quantile",
        aggfunc = "median"
    )
)

# Reorder to match model/layer ordering from wide_consistent_attentions
reordered_index = reorder_multindex_by_categorical_and_numeric(
    conditional_quantiles.index,
    categorical_order=model_order,
    categorical_level=0,  # query_model
    numeric_level=1       # query_layer
)

# Get categorical order (model order) from reference
# Reorder index and columns
conditional_quantiles = conditional_quantiles.reindex(
    index=reordered_index,
    columns=reordered_index
)

# mask within-model block diagonal
model_col_values = conditional_quantiles.columns.get_level_values(0)

# Create mask where models match (True = mask/hide these values)
mask = model_col_values.to_numpy()[:, None] == model_col_values.to_numpy()[None, :]


In [ ]:
plot_heatmap(
    conditional_quantiles,
    row_labels = model_layer_labels,
    suptitle="Cross-model x layer attention consistency",
    title="Median quantiles of top attention pairs from another\nmodel x layer. Within-model attention is masked.",
    xlabel = "Evaluated model & layer",
    ylabel = "TopK model & layer",
    cmap='viridis',
    cbar=True,
    fmt='.2f',
    vmax=1,
    vmin=-0,
    cbar_label='Median rank',
    mask = mask,
    mask_upper_triangle=False,
    mask_color='lightgray',
    annot=False,
    square=True,
    suptitle_size=18,
    title_size=16,
    axis_title_size=16,
    tick_label_size=5,
    title_fontstyle="italic",
    title_fontweight="normal",
    figsize=(9, 9)
)

plt.show()

## Across models (comparing cross-layer model consensus attentions)

In [ ]:
def calculate_cross_model_consensus_attention_correlation(consensus_top_k):

    # look at the top attentions for each model/layer across all models and layers
    wide_consensus_top_k = (
        consensus_top_k.pivot(index = [FM_EDGELIST.FROM_GENE, FM_EDGELIST.TO_GENE], columns = FM_EDGELIST.MODEL, values = FM_EDGELIST.ATTENTION)
        .reindex(columns=model_order)
    )

    return compute_correlation_matrix(wide_consensus_top_k.to_numpy())

consensus_top_k = comparisons["cross_model_consensus_top_attentions"]
if "category" in consensus_top_k.columns:
    per_category_corr = consensus_top_k.groupby("category").apply(calculate_cross_model_consensus_attention_correlation)
    cross_model_attention_corr = np.median(np.stack(per_category_corr.values), axis=0)
else:
    cross_model_attention_corr = calculate_cross_model_consensus_attention_correlation(cross_model_top_attentions)

In [ ]:
plot_heatmap(
    cross_model_attention_corr,
    row_labels = consensus_top_k["model"].unique(),
    suptitle="Cross-model attention consistency",
    title=f"Correlation of attention logits across the union of\nmodel-level top {TOP_K} gene x gene attention pairs",
    cmap=bwy,
    cbar=False,
    fmt='.2f',
    vmax=0.5,
    vmin=-0.5,
    mask_upper_triangle=True,
    annot=True,
    square=True,
    suptitle_size=18,
    title_size=16,
    axis_title_size=16,
    title_fontstyle="italic",
    title_fontweight="normal",
    figsize=(9, 9)
)

plt.show()

In [ ]:
conditional_quantiles = (
    comparisons["cross_model_consensus_top_attentions_rank_agreement"]
    .pivot_table(
        index = ["query_model"],
        columns = ["eval_model"],
        values = "median_quantile",
        aggfunc = "median"
    )
    .reindex(index=model_order, columns=model_order)
)

plot_heatmap(
    conditional_quantiles,
    row_labels = conditional_quantiles.columns.tolist(),
    column_labels = conditional_quantiles.index.tolist(),
    suptitle="Cross-model attention consistency",
    title=f"Median quantiles of attention scores in evaluated model\ngiven that they were the top {TOP_K} attention pairs\nin another model",
    xlabel = "Evaluated model",
    ylabel = f"Top {TOP_K} model",
    cmap='viridis',
    cbar=False,
    fmt='.2f',
    vmax=1,
    vmin=0,
    cbar_label='Rank aggreement',
    mask_upper_triangle=False,
    annot=True,
    square=True,
    suptitle_size=18,
    title_size=16,
    axis_title_size=16,
    title_fontstyle="italic",
    title_fontweight="normal",
    figsize=(9, 9)
)

plt.show()

## Summarize the cross-model coherence of individual layers

In [ ]:
from napistu_torch.evaluation.manager import RemoteEvaluationManager
from napistu.network.constants import NAPISTU_GRAPH_VERTICES, NAPISTU_GRAPH_EDGES
import torch
from napistu_torch.utils.tensor_utils import compute_cosine_distances_torch


def summarize_cross_model_attention_coherence(
    model_x_layer_rank_agreement: pd.DataFrame,
    model_sources: pd.DataFrame
) -> pd.Series:
    """
    Summarize each layer's cross-source attention coherence by filtering to only include layers from different sources and aggregating the quantile deviations from 0.5

    Parameters
    ----------
    model_x_layer_rank_agreement : pd.DataFrame
        A dataframe with the median quantile of the top K set for each model, layer, and category
    model_sources : pd.DataFrame
        A dataframe with the model and model_category for each model

    Returns
    -------
    pd.Series
        A series with the log10 quantile deviation from 0.5 for each model, layer, and category
    """
        

    return (
        model_x_layer_rank_agreement
        .merge(
            model_sources.rename(columns = {"model" : "query_model", "model_category" : "query_model_category"}),
            on = ["query_model"], how = "left"
        )
        .merge(
            model_sources.rename(columns = {"model" : "eval_model", "model_category" : "eval_model_category"}),
            on = ["eval_model"], how = "left"
        )
        .query("query_model_category != eval_model_category")
        # summarize how much less than 0.5 (i.e., the median quantile) the top K set is across all layers
        .assign(
            log_quantile=lambda x: -np.log10(x["median_quantile"].clip(upper=0.5) * 2)
        )
        .groupby(["query_model", "query_layer", "category"])
        ["log_quantile"]
        .mean()
        .sort_values(ascending = False)
    )

def add_vertex_names_to_edgelist(
    edgelist: pd.DataFrame,
    gene_to_vertex_map: pd.DataFrame
) -> pd.DataFrame:
    """
    Add from_vertex and to_vertex attributes by mapping an edgelist with ensembl gene include_scgpt

    Parameters
    ----------
    edgelist : pd.DataFrame
        An edgelist with from_gene and to_gene columns
    gene_to_vertex_map : pd.DataFrame
        A dataframe with ensembl_gene and name columns

    Returns
    -------
    pd.DataFrame
        An edgelist with from_vertex and to_vertex columns
    """

    top_k_with_ids = (
        edgelist
        .merge(
            (
                gene_to_vertex_map
                .rename(columns = {ONTOLOGIES.ENSEMBL_GENE : FM_EDGELIST.FROM_GENE, NAPISTU_GRAPH_VERTICES.NAME : "from_vertex"})
                .drop(columns = [SBML_DFS.S_ID])
            ),
            on = FM_EDGELIST.FROM_GENE,
            how = "left"
        )
        .merge(
            gene_to_vertex_map
            .rename(columns = {ONTOLOGIES.ENSEMBL_GENE : FM_EDGELIST.TO_GENE, NAPISTU_GRAPH_VERTICES.NAME : "to_vertex"})
            .drop(columns = [SBML_DFS.S_ID]),
            on = FM_EDGELIST.TO_GENE,
            how = "left"
        )
    )

    # drop NAs and log
    invalid_edges = top_k_with_ids.isna().any(axis = 1)
    if invalid_edges.sum() > 0:
        percent_invalid = (invalid_edges.sum() / len(top_k_with_ids)) * 100
        logger.warning(f"Dropping {invalid_edges.sum()} edges ({percent_invalid:.2f}%) which could not be mapped to vertices")

    return top_k_with_ids.loc[~invalid_edges]


def top_attention_to_napistu_edgelist(cross_model_top_attentions: pd.DataFrame, gene_to_vertex_map: pd.DataFrame) -> pd.DataFrame:
    """
    Convert top attention pairs to an edgelist with from and to vertex names

    Parameters
    ----------
    cross_model_top_attentions : pd.DataFrame
        The top attention pairs
    gene_to_vertex_map : pd.DataFrame
        The gene to vertex map

    Returns
    -------
    pd.DataFrame
        The edgelist with from and to vertex names
    """
    # look at the topK attentions for the selected layers
    top_k_attention = (
        cross_model_top_attentions
        .query("attention_rank <= @TOP_K")
        .set_index(["model", "layer"])
        .sort_values("attention_rank")
        .sort_index()
    )

    # filter to just the from-to genes for the top attentions
    top_k_attention_distinct_edges = top_k_attention[["from_gene", "to_gene"]].drop_duplicates()

    top_k_attention_edgelist = add_vertex_names_to_edgelist(
        top_k_attention_distinct_edges,
        gene_to_vertex_map
    )

    return top_k_attention_edgelist


def top_attention_to_napistu_edgelist(
    cross_model_top_attentions: pd.DataFrame,
    gene_to_vertex_map: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Convert top attention pairs to an edgelist with from and to vertex names

    Parameters
    ----------
    cross_model_top_attentions : pd.DataFrame
        The top attention pairs
    gene_to_vertex_map : pd.DataFrame
        The gene to vertex map

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame]
        The top attention pairs and the edgelist
    """

    # look at the topK attentions for the selected layers
    top_k_attention = (
        cross_model_top_attentions
        .query("attention_rank <= @TOP_K")
        .set_index(["model", "layer"])
        .sort_values("attention_rank")
        .sort_index()
    )

    # filter to just the from-to genes for the top attentions
    top_k_attention_distinct_edges = top_k_attention[["from_gene", "to_gene"]].drop_duplicates()

    top_k_attention_edgelist = add_vertex_names_to_edgelist(
        top_k_attention_distinct_edges,
        gene_to_vertex_map
    )

    return top_k_attention, top_k_attention_edgelist

def calculate_background_edgelist_metrics(
    napistu_data: "NapistuData",
    edge_prediction_task: "EdgePredictionTask",
    vertex_names: List[str],
) -> Tuple[float, float]:
    """
    Calculate background edgelist metrics

    Parameters
    ----------
    napistu_data : NapistuData
        The NapistuData object
    edge_prediction_task : EdgePredictionTask
        The GNN model that will be used to predict edges
    vertex_names : List[str]
        The names of the vertices

    Returns
    -------
    Tuple[float, float]
        The background edge rate and the average edge score
    """

    # Get the gene vertex indices
    gene_vertices = set(napistu_data.get_vertex_indices(vertex_names))

    # edge_index shape: [2, num_edges]
    src = napistu_data.edge_index[0]
    dst = napistu_data.edge_index[1]

    # Mask edges where both endpoints are in the gene vertex set
    gene_vertices_tensor = torch.tensor(list(gene_vertices), dtype=torch.long)

    src_mask = torch.isin(src, gene_vertices_tensor)
    dst_mask = torch.isin(dst, gene_vertices_tensor)

    both_mask = src_mask & dst_mask
    num_edges = both_mask.sum().item()

    # density of edges among genes in the vocabulary
    background_edge_rate = num_edges / (len(gene_vertices)-1) ** 2

    # calculate the average edge score for all edges in the universe
    all_pairs = pd.MultiIndex.from_product(
        [vertex_names, vertex_names], names=[NAPISTU_GRAPH_EDGES.FROM, NAPISTU_GRAPH_EDGES.TO]
    ).to_frame(index=False)
    all_pairs = all_pairs[all_pairs[NAPISTU_GRAPH_EDGES.FROM] != all_pairs[NAPISTU_GRAPH_EDGES.TO]]
    # randomly subset since we are just interested in the average score
    all_pairs = all_pairs.sample(10000)

    background_edge_score = edge_prediction_task.predict_edge_scores(
        napistu_data,
        napistu_data.get_edge_indices(all_pairs, NAPISTU_GRAPH_EDGES.FROM, NAPISTU_GRAPH_EDGES.TO)
    ).mean().numpy()

    return background_edge_rate, background_edge_score


def compare_attention_and_napistu_graphs(
    top_k_attention: pd.DataFrame,
    top_k_attention_edgelist: pd.DataFrame,
    napistu_data: "NapistuData",
    edge_prediction_task: "EdgePredictionTask",
    model_order: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compare the attention of a model with the edges in the NapistuData object

    Parameters
    ----------
    top_k_attention : pd.DataFrame
        The top K attention pairs for every model, layer and category
    top_k_attention_edgelist : pd.DataFrame
        The distinct top K attention pairs with Napistu vertex IDs added
    napistu_data : NapistuData
        The NapistuData object
    edge_prediction_task : "EdgePredictionTask"
        The edge prediction task
    model_order : List[str]
        The order of the models

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame]
        The direct edge rate and the average edge prediction for every model, layer and category
    """


    # find the indices of the edges in the NapistuData object
    edge_indices = napistu_data.get_edge_indices(top_k_attention_edgelist, "from_vertex", "to_vertex")
    predictions = edge_prediction_task.predict_edge_scores(
        data=napistu_data,
        edge_index=edge_indices,
    )

    # add GNN predictions and whether the edge exists in the NapistuData object
    top_k_attention_edgelist["prediction"] = predictions
    top_k_attention_edgelist["direct_edge_exists"] = napistu_data.has_edges(edge_indices)

    top_k_attention_w_metrics = (
        top_k_attention
        .reset_index()
        .merge(
            top_k_attention_edgelist[["from_gene", "to_gene", "prediction", "direct_edge_exists"]],
            on = ["from_gene", "to_gene"],
            how = "left"
        )
    )

    direct_edge_rate = (
        top_k_attention_w_metrics.value_counts(["model", "layer", "category", "direct_edge_exists"])
        .reset_index()
        .pivot(index = ["model", "layer", "category"], columns = "direct_edge_exists", values = "count")
        .fillna(0)
        .assign(true_fraction = lambda x: x[True] / x.sum(axis = 1))
        .reset_index()
        .assign(model_name = lambda x: pd.Categorical(
            x['model'], 
            categories=model_order, 
            ordered=True
        ))
    )

    average_edge_prediction = (
        top_k_attention_w_metrics
        .groupby(["model", "layer", "category"])["prediction"]
        .median()
        .reset_index()
        .assign(model_name = lambda x: x["model"])
        .assign(model_name = lambda x: pd.Categorical(
            x['model_name'], 
            categories=model_order, 
            ordered=True
        ))
    )

    return direct_edge_rate, average_edge_prediction


def get_model_label_maps(model_metadata_summary):
    model_type_map = model_metadata_summary.set_index('model')['model type'].to_dict()
    model_variant_map = model_metadata_summary.set_index('model')['model variant'].to_dict()
    return model_type_map, model_variant_map

def plot_metric_by_layer(
    data,
    y_col,
    background_value,
    ylabel,
    model_order,
    model_metadata_summary,
    figsize=(14, 4),
    bar_width=0.6,
    group_gap=2,
):

    model_type_map, model_variant_map = get_model_label_maps(model_metadata_summary)
    fig, ax = plt.subplots(figsize=figsize)

    x_pos_map = {}
    current_x = 0
    model_label_positions = {}

    for model in model_order:
        model_data = data[data['model_name'] == model]
        layers = sorted(model_data['layer'].unique())
        model_start = current_x

        for layer in layers:
            x_pos_map[(model, layer)] = current_x
            current_x += 1

        model_label_positions[model] = (model_start + current_x - 1) / 2
        current_x += group_gap

    for model in model_order:
        model_data = data[data['model_name'] == model]

        for layer, group in model_data.groupby('layer'):
            x = x_pos_map[(model, layer)]
            mean_val = group[y_col].mean()
            min_val = group[y_col].min()
            max_val = group[y_col].max()

            ax.bar(x, mean_val, width=bar_width, color='#aaaaaa', alpha=0.8)
            ax.plot([x, x], [min_val, max_val], color='black', linewidth=1, zorder=5)
            ax.plot([x - 0.15, x + 0.15], [min_val, min_val], color='black', linewidth=1, zorder=5)
            ax.plot([x - 0.15, x + 0.15], [max_val, max_val], color='black', linewidth=1, zorder=5)

    ax.set_xticks([x_pos_map[(m, l)] for m in model_order
                   for l in sorted(data[data['model_name'] == m]['layer'].unique())])
    ax.set_xticklabels([str(l) for m in model_order
                        for l in sorted(data[data['model_name'] == m]['layer'].unique())],
                       fontsize=7)

    ax.axhline(background_value, color='gray', linestyle=':', linewidth=1.5, zorder=0)
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()

    y_max = ax.get_ylim()[1]
    y_type = y_max * 1.12
    y_variant = y_max * 1.04

    seen_types = {}
    for model, x_center in model_label_positions.items():
        model_type = model_type_map.get(model, model)
        variant = model_variant_map.get(model)

        if model_type not in seen_types:
            seen_types[model_type] = []
        seen_types[model_type].append(x_center)

        if pd.notna(variant):
            ax.text(x_center, y_variant, variant, ha='center', va='bottom',
                    fontsize=8, color='#555555')

    for model_type, positions in seen_types.items():
        x_center = np.mean(positions)
        ax.text(x_center, y_type, model_type, ha='center', va='bottom',
                fontsize=10, fontweight='bold')

    return fig, ax

def summarize_edgelist_cosine_similarity(
    top_k_attention: pd.DataFrame,
    top_k_attention_edgelist: pd.DataFrame,
    napistu_data: "NapistuData",
    napistu_gnn: "NapistuGNN",
    model_order: List[str]
) -> Tuple[pd.DataFrame, float]:
    """
    Summarize an edgelist's cosine similarity based on the vertex embeddings

    Parameters
    ----------
    top_k_attention : pd.DataFrame
        The top K attention pairs for every model, layer and category
    top_k_attention_edgelist : pd.DataFrame
        The distinct top K attention pairs with Napistu vertex IDs added
    napistu_data : "NapistuData"
        The NapistuData object
    napistu_gnn : "NapistuGNN"
        The NapistuGNN object
    model_order : List[str]
        The order of the models

    Returns
    -------
    Tuple[pd.DataFrame, float]
        The average cosine similarity per model, layer and category and the background cosine similarity
    """

    # load the model's vertex embeddings

    vertex_names = pd.concat([top_k_attention_edgelist["from_vertex"], top_k_attention_edgelist["to_vertex"]]).unique().tolist()
    vertex_indices = napistu_data.get_vertex_indices(vertex_names)

    # extract the embedding for just the vertices of interest and calculate the gene x gene cosine similarity
    cosine_sim = 1 - compute_cosine_distances_torch(napistu_gnn.get_embeddings(napistu_data)[vertex_indices])

    # create a lookup table from vertex names to their indices in the 
    vertex_to_idx = {name: i for i, name in enumerate(napistu_data.get_vertex_names()[vertex_indices])}

    from_idx = top_k_attention_edgelist["from_vertex"].map(vertex_to_idx)
    to_idx = top_k_attention_edgelist["to_vertex"].map(vertex_to_idx)

    # Select values from the matrix using positional indexing
    working_top_k_attention_edgelist = top_k_attention_edgelist.copy()
    working_top_k_attention_edgelist["cosine_sim"] = cosine_sim[
        from_idx.values,
        to_idx.values
    ]

    average_cosine_sim = (
        top_k_attention.reset_index()
        .merge(working_top_k_attention_edgelist, on = ["from_gene", "to_gene"], how = "inner")
        # calculate the median cosine similarity
        .groupby(["model", "layer", "category"])["cosine_sim"].median()
        .reset_index()
        .assign(model_name = lambda x: pd.Categorical(
            x['model'], 
            categories=model_order, 
            ordered=True
        ))
    )

    background_cosine_sim = cosine_sim.mean()

    return average_cosine_sim, background_cosine_sim


In [ ]:
model_categories = pd.DataFrame([
    {"model type": "scGPT", "model_category" : "scGPT"},
    {"model type": "scPRINT", "model_category" : "scPRINT"},
    {"model type": "scFoundation", "model_category" : "genbioAI"},
    {"model type": "AIDOCell", "model_category" : "genbioAI"},
])

# load the Napistu Edge Prediction MLP
evaluation_manager = RemoteEvaluationManager.from_huggingface(
    "seanhacks/edge_prediction_mlp_256e",
    data_store_dir = STORE_DIR
)
napistu_gnn = evaluation_manager.load_model_from_checkpoint()
napistu_data = evaluation_manager.load_napistu_data()

In [ ]:
model_x_layer_rank_agreement = model_summaries[True]["comparisons"]["cross_model_x_layer_rank_agreement"]
cross_model_x_layer_top_attentions = model_summaries[True]["comparisons"]["cross_model_x_layer_top_attentions"]
model_comparison_metadata = model_summaries[True]["model_comparison_metadata"]
model_metadata_summary = model_comparison_metadata["model_metadata_summary"]
gene_ids = model_comparison_metadata["gene_ids"]
model_order = model_comparison_metadata["model_order"]

gene_to_vertex_map = map_identifiers_to_vertex_names(gene_ids, species_identifiers, name_to_sid_map)

In [ ]:
model_sources = model_categories.merge(model_metadata_summary, on = ["model type"], how = "inner")[["model", "model_category"]]
# summarize how much a layer's attention aligns with low quantiles in other model source's layers
cross_model_attention_coherence = summarize_cross_model_attention_coherence(model_x_layer_rank_agreement, model_sources)

# filter cross_model_x_layer_top_attentions to just the within model topK and return the distinct edges with their Napistu vertex IDs added
top_k_attention, top_k_attention_edgelist = top_attention_to_napistu_edgelist(cross_model_x_layer_top_attentions, gene_to_vertex_map)

# summarize the direct edge rate and average edge prediction for every model, layer and category
direct_edge_rate, average_edge_prediction = compare_attention_and_napistu_graphs(
    top_k_attention,
    top_k_attention_edgelist,
    napistu_data,
    napistu_gnn.task,
    model_order
)

# find the average # of connections and average edge score among the genes in the shared vocabulary
background_edge_rate, background_edge_score = calculate_background_edgelist_metrics(
    napistu_data,
    napistu_gnn.task,
    gene_to_vertex_map["name"]
)

In [ ]:
fig, ax = plot_metric_by_layer(
    data=direct_edge_rate,
    y_col='true_fraction',
    background_value=background_edge_rate,
    ylabel='True Fraction',
    model_order=model_order,
    model_metadata_summary=model_metadata_summary,
)

fig, ax = plot_metric_by_layer(
    data=average_edge_prediction,
    y_col='prediction',
    background_value=background_edge_score,
    ylabel='Median Edge Score',
    model_order=model_order,
    model_metadata_summary=model_metadata_summary,
)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Build both dataframes
df1 = average_edge_prediction.merge(
    cross_model_attention_coherence,
    left_on=["model", "layer", "category"],
    right_index=True
)

df2 = direct_edge_rate.merge(
    cross_model_attention_coherence,
    left_on=["model", "layer", "category"],
    right_index=True
)

# Shared color map across both plots
models = pd.concat([df1["model"], df2["model"]]).unique()
colors = cm.tab10(np.linspace(0, 1, len(models)))
color_map = dict(zip(models, colors))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df, ylabel in zip(
    axes,
    [df1, df2],
    ["prediction", "true_fraction"]
):
    for model, group in df.groupby("model"):
        ax.scatter(
            group["log_quantile"],
            group[ylabel],
            color=color_map[model],
            label=model, alpha=0.6, s=20
            )
    ax.set_xlabel("log_quantile")
    ax.set_ylabel(ylabel)

# Single shared legend to the right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.02, 0.5), loc="center left", frameon=False)

plt.tight_layout()
plt.show()

## What do the high scoring edges represent?

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


def plot_stacked_histogram(ax, df, score_col, x_min, x_max, n_bins=40):
    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    width = bin_edges[1] - bin_edges[0]

    no_edge = df[~df["direct_edge_exists"]][score_col]
    has_edge = df[df["direct_edge_exists"]][score_col]

    no_edge_counts, _ = np.histogram(no_edge, bins=bin_edges)
    has_edge_counts, _ = np.histogram(has_edge, bins=bin_edges)

    ax.bar(bin_centers, has_edge_counts, width=width, label="Direct edge", alpha=0.8, color="#444444")
    ax.bar(bin_centers, no_edge_counts, width=width, bottom=has_edge_counts, label="No direct edge", alpha=0.8, color="#cccccc")
    
    medians = [
        (False, "No edge median",    "--"),
        (True,  "Direct edge median", "-."),
    ]

    meds = [(df[df["direct_edge_exists"] == exists][score_col].median(), label, ls)
            for exists, label, ls in medians]
    meds_sorted = sorted(meds, key=lambda x: x[0])

    y_positions = [0.97, 0.83] if len(meds_sorted) == 2 else [0.97]

    for (med, label, ls), y_pos in zip(meds_sorted, y_positions):
        ax.axvline(med, linestyle=ls, linewidth=1.5, color = "gray")
        ax.text(
            med + 0.01, y_pos, f"{label}: {med:.2f}",
            transform=ax.get_xaxis_transform(),
            fontsize=8, va="top",
        )

    ax.set_xlabel(score_col.capitalize())
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)


fig, ax = plt.subplots(figsize=(8, 5))

plot_stacked_histogram(ax, top_k_attention_edgelist, "prediction", x_min=0, x_max=1)
ax.set_title("GNN edge prediction score by edge existence")

plt.tight_layout()
plt.show()

In [ ]:
# if scores are low, then are high attention edges preferentially between distant genes?

average_embedding_cosine_similarity, background_embedding_cosine_similarity = summarize_edgelist_cosine_similarity(
    top_k_attention,
    top_k_attention_edgelist,
    napistu_data,
    napistu_gnn,
    model_order
)

In [ ]:
fig, ax = plot_metric_by_layer(
    data=average_embedding_cosine_similarity,
    y_col='cosine_sim',
    background_value=background_embedding_cosine_similarity,
    ylabel='Average vertex embedding cosine similarity\nbetween from-to high-attention pairs',
    model_order=model_order,
    model_metadata_summary=model_metadata_summary,
)